In [ ]:
# === Common notebook preamble (import llm_math package) ===
import sys
from pathlib import Path

# Candidate paths to locate the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add all parent directories of the current working directory to candidates (when run from notebooks/ folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Attempt to import llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] Failed to load llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === preamble end ===

# Ch 00. This Book's Learning Approach and Environment Setup

> **Learning Objectives**
> - Can explain why mathematics is vital in the LLM era
> - Understand the book's top-down design and study methodology
> - Can freely switch between CPU/GPU runtimes on Colab
> - Can use the shared utilities package `llm_math`

## 0.1 Why "Math First" for LLMs?

ChatGPT, GPT-4, LLaMA, Claude... LLMs (large language models) have reshaped the world since 2022.
But most developers merely call APIs without grasping what happens inside the model.

This book aims to **start from a single equation and build LMs from scratch**.
- Why does matrix multiplication $AB$ form the core of attention?
- How does softmax $\sigma(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$ create a probability distribution?
- How is backpropagation mechanically applied using the chain rule $\frac{d}{dx}f(g(x)) = f'(g(x)) g'(x)$?

Answering these questions lets you move beyond *using* LLMs to *understanding* and *creating* them.

## 0.2 This Book's Design Principles

```
Mathematical definition → Intuition (diagram) → Formula derivation → Code implementation → Benchmark → Application
```
Every chapter follows this 6-step structure.

## 0.3 Environment Setup

Assume you are running this notebook in Colab. Execute the cell below to inspect your environment.


## Loading the `llm_math` package

All notebooks in this book use the shared utility package `llm_math`. It is automatically imported in the first code cell. (Try it yourself!)

In [ ]:
# Check the llm_math state loaded from the preamble
if _LLM_MATH_OK:
    print("✓ llm_math 패키지 로드 성공")
    print(f"  - viz: Visualization 헬퍼 ({viz.__name__})")
    print(f"  - bench: CPU vs GPU 벤치마크 헬퍼 ({bench.__name__})")
    print(f"  - data: 미니 Data셋 로더 ({data.__name__})")
else:
    print("⚠ 로드 실패. 아래 명령으로 레포를 클론하세요:")
    print("  !git clone https://bit.ly/4f1YusE")
    print("  %cd llm-math-book")
    print("  !bash colab_setup.sh")

## 0.4 First Benchmark: A quick taste of CPU vs. GPU with a small matrix multiply

One of the central themes of this book is **the intuition behind the speed difference between CPU and GPU**. Let’s start with a very simple matrix multiplication.


In [ ]:
# First Benchmark: CPU vs GPU Matrix multiplication
import time
import numpy as np

def bench_np_matmul(n, repeat=5):
    """NumPy Matrix multiplication Time Measurement (CPU)."""
    A = np.random.randn(n, n).astype(np.float32)
    B = np.random.randn(n, n).astype(np.float32)
    # warm-up
    for _ in range(2):
        _ = A @ B
    # measurement
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        _ = A @ B
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000)
    return np.mean(times), np.std(times)

# Matrix Size-wise Measurement
for n in [64, 256, 1024, 2048]:
    mean_ms, std_ms = bench_np_matmul(n, repeat=3)
    print(f"  n={n:5d}: {mean_ms:8.3f} +- {std_ms:6.3f} ms")

In [ ]:
# Compare PyTorch CPU vs GPU Matrix Multiplication
try:
    import torch
    print("PyTorch CPU vs GPU Matrix Multiplication Comparison:\n")
    print(f"{'n':>6} | {'CPU (ms)':>12} | {'GPU (ms)':>12} | {'Speedup':>10}")
    print('-' * 50)
    for n in [64, 256, 1024, 2048]:
        # CPU
        A_cpu = torch.randn(n, n)
        B_cpu = torch.randn(n, n)
        for _ in range(2): _ = A_cpu @ B_cpu  # warmup
        t0 = time.perf_counter()
        for _ in range(3):
            _ = A_cpu @ B_cpu
        cpu_ms = (time.perf_counter() - t0) / 3 * 1000

        if torch.cuda.is_available():
            A_gpu = A_cpu.cuda()
            B_gpu = B_cpu.cuda()
            for _ in range(2): _ = A_gpu @ B_gpu  # warmup
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(3):
                _ = A_gpu @ B_gpu
            torch.cuda.synchronize()
            gpu_ms = (time.perf_counter() - t0) / 3 * 1000
            speedup = cpu_ms / gpu_ms
            print(f"{n:>6} | {cpu_ms:>12.3f} | {gpu_ms:>12.3f} | {speedup:>9.2f}x")
        else:
            print(f"{n:>6} | {cpu_ms:>12.3f} | {'N/A':>12} | {'N/A':>10}")
    if not torch.cuda.is_available():
        print("\n⚠ GPU가 없습니다. Colab 런타임을 T4 GPU로 전환하세요: Runtime → Change runtime type")
except ImportError:
    print("PyTorch가 없어 이 셀은 건너뜁니다.")

# 0.5 Benchmarking Methodology

All benchmarks in this book adhere to the following principles:

1. **Warm-up**: The first 2–3 executions, which suffer from caching or compilation effects, are omitted from measurement.
2. **Repeated Measurement**: Report the mean and standard deviation after 5–10 repetitions.
3. **Synchronization**: For GPU measurements, invoke `torch.cuda.synchronize()` to ensure all asynchronous operations are completed before recording times.
4. **Memory Tracing**: For GPU, measure peak memory usage using `torch.cuda.max_memory_allocated()`.
5. **Identical Data**: Both CPU and GPU experiments use the same initialization seed.

A helper function automating this procedure is available in `llm_math.bench`.


In [ ]:
# llm_math.bench helper usage example
import torch
from llm_math.bench import time_fn, format_results_table, get_device

device = get_device()
print(f"Current device: {device}")

def matmul_op(A, B):
    return A @ B

results = {}
for dev_name in ['cpu'] + (['cuda'] if torch.cuda.is_available() else []):
    n = 1024
    A = torch.randn(n, n, device=dev_name)
    B = torch.randn(n, n, device=dev_name)
    res = time_fn(matmul_op, A, B, device=dev_name, warmup=2, repeat=5)
    results[dev_name.upper()] = res

print(format_results_table(results, title="Matrix multiplication (n=1024) benchmark"))

# 0.6 Practice Problems and Solution Notebook

End-of-chapter exercises are provided. Solutions are located in `solutions/chXX_solutions.ipynb`.

## 0.7 What’s Next

In **Chapter 01. Vectors and Space**, we explore the vector that underlies everything in LLMs.
Embeddings, attention, gradients — all expressed in the language of vectors and space.

---

### Summary
- Understanding LLMs internally requires mathematics (linear algebra, calculus, probability).
- This book adopts a bottom‑up approach: mathematics → neural networks → attention → transformers → LLMs.
- All notebooks can switch between CPU and GPU (starting from Colab page 9).
- We rely on a shared utility package `llm_math` for simulations.

